# Ordered Logistic Regression Results for Adoption Predictors in Rangeland Management (FAIR^2) Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll print out the names and `@id` of all available record sets. Each record set contains fields and columns describing the data. We will further list out the available fields for each record set, referencing all entities by their `@id`.

In [ ]:
# List all record sets in the dataset
record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    print("Record sets found in the dataset:")
    for rs in record_sets:
        print(f"- Name: {rs.name}")
        print(f"  @id: {rs.id}")

    # Show fields for each record set
    print("\nFields available in each record set:")
    for rs in record_sets:
        print(f"\nRecordSet: {rs.name} (@id: {rs.id})")
        if not rs.fields:
            print("  (No fields listed in this RecordSet.)")
        else:
            for field in rs.fields:
                print(f"- Field Name: {field.name}")
                print(f"  Field @id: {field.id}")
                if hasattr(field, 'column') and field.column:
                    col = field.column
                    print(f"    Column @id: {col.id}")
            print("")

### Example Record Preview
Let's preview records from each record set (if any are available).

In [ ]:
# Preview records from every record set by @id
for rs in record_sets:
    print(f"\nFirst 2 records from RecordSet '{rs.name}' (@id: {rs.id}):")
    try:
        for i, record in enumerate(dataset.records(record_set=rs.id)):
            if i > 1:
                break
            print(record)
    except Exception as e:
        print(f"  [Error loading records: {e}]")

## 3. Data Extraction
Load data from each available record set into a pandas DataFrame using the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set into pandas DataFrames.
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for rs in record_sets:
    print(f"Loading DataFrame for RecordSet '{rs.name}' (@id: {rs.id})...")
    try:
        records = list(dataset.records(record_set=rs.id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs.id] = df
            print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")
        else:
            print("  No records found.")
    except Exception as e:
        print(f"  Error reading records: {e}")

# For demonstration, pick the first non-empty record set
main_rs_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        main_rs_id = rs_id
        break
if main_rs_id:
    print(f"\nColumns in main record set (@id: {main_rs_id}):")
    print(dataframes[main_rs_id].columns.tolist())
    print(dataframes[main_rs_id].head())
else:
    print("\nNo data found in any record set.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations such as removing outliers, transforming distributions, or aggregating by key attributes will prepare data for further analysis.

**All references are by `@id`.**

In [ ]:
# Select the main DataFrame for EDA
if main_rs_id:
    df = dataframes[main_rs_id].copy()
    columns = df.columns.tolist()
    print(f"Columns in main DataFrame (@id: {main_rs_id}): {columns}")
    # Try to select a numeric field by inspecting dtypes
    numeric_field_id = None
    for col in columns:
        # Try conversion to float to detect numeric columns
        try:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
        except Exception:
            continue
    if numeric_field_id is not None:
        print(f"\nSelected numeric field for analysis: {numeric_field_id}")
        
        threshold = df[numeric_field_id].mean()  # Use mean as threshold for demonstration
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df[[numeric_field_id]].head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try group-by on a likely categorical field (choose the first non-numeric column)
        group_field_id = None
        for col in columns:
            if col != numeric_field_id and (
                df[col].dtype == object or not pd.api.types.is_numeric_dtype(df[col])):
                group_field_id = col
                break
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("\nNo suitable non-numeric group field found.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only visualize if data is available
if main_rs_id and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If group_field_id available, plot mean of numeric value by category
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10, 5))
        order = df[group_field_id].value_counts().index[:10]
        sns.barplot(
            data=df[df[group_field_id].isin(order)],
            x=group_field_id, y=numeric_field_id, ci=None, order=order
        )
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=30, ha='right')
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion

In this notebook, we demonstrated how to use the `mlcroissant` library to:
- Load and inspect a FAIR^2 dataset defined by a Croissant schema at the URL: `https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`
- List available record sets and their fields by `@id`
- Load records into pandas DataFrames
- Perform simple exploratory data analysis and visualize data distributions

Key observations and findings will depend on the specific dataset contents. This template enables FAIR exploration for any Croissant-compliant dataset using entity IDs for reliable, explicit referencing.